# Assignment 03 — Image Classification Using Pretrained CNN Architectures

## Title
**Implementation and Performance Comparison of ResNet50 and GoogLeNet/Inception (InceptionV3) for Image Classification**

## Objective
Implement and compare **ResNet50** and **InceptionV3** using **transfer learning** on **CIFAR-10**.

## Deliverables
- Training code (TensorFlow/Keras)
- Accuracy/Loss curves
- Confusion matrix + classification report
- Comparison: accuracy, loss, training time, model size
- Standalone report exported to Google Drive (`report.html` + `report.md`)

---

## Colab checklist
1. `Runtime → Change runtime type → GPU (T4)`
2. `Runtime → Restart runtime` (recommended before a clean run)
3. `Runtime → Run all`

All outputs are saved to Google Drive folder: `MyDrive/COMP-443/Assignment_03_ResNet_vs_Inception/`.


In [ ]:
# =============================
# 1) Setup
# =============================
NOTEBOOK_VERSION = 'assignment03_colab_ready_v3_2026-04-24'
print('Notebook version:', NOTEBOOK_VERSION)

import os
import time
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


## 1.1) Dependencies (Colab)
Colab usually includes these. If imports fail, run the install cell.


In [ ]:
# If running in Colab and you get import errors, uncomment:
# !pip -q install -U scikit-learn pandas

from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd


## 1.2) Google Drive (save everything)


In [ ]:
# Mount Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print('Not running in Colab (Drive mount skipped):', e)

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = f"{DRIVE_ROOT}/COMP-443/Assignment_03_ResNet_vs_Inception"
LOCAL_DIR = '/content/assignment03_outputs'

OUTPUT_DIR = PROJECT_DIR if IN_COLAB else LOCAL_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 1.3) Configuration
Use `FAST_MODE=True` for a quicker first run.


In [ ]:
# =============================
# Config
# =============================
FAST_MODE = False          # True: fewer epochs + smaller Inception input
CACHE_DATASETS = True      # caches after preprocessing (RAM)
USE_MIXED_PRECISION = True # good speedup on T4

BATCH_SIZE = 64 if not FAST_MODE else 32
EPOCHS_HEAD = 8 if not FAST_MODE else 4
FINE_TUNE = True
FINE_TUNE_EPOCHS = 7 if not FAST_MODE else 3

RESNET_IMAGE_SIZE = (224, 224)
INCEPTION_IMAGE_SIZE = (299, 299) if not FAST_MODE else (224, 224)

UNFREEZE_LAST_N_RESNET = 40
UNFREEZE_LAST_N_INCEPTION = 60

print('FAST_MODE:', FAST_MODE)
print('BATCH_SIZE:', BATCH_SIZE)
print('EPOCHS_HEAD:', EPOCHS_HEAD, 'FINE_TUNE_EPOCHS:', FINE_TUNE_EPOCHS)
print('RESNET_IMAGE_SIZE:', RESNET_IMAGE_SIZE)
print('INCEPTION_IMAGE_SIZE:', INCEPTION_IMAGE_SIZE)


In [ ]:
# Mixed precision (optional but recommended on T4)
if USE_MIXED_PRECISION:
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision enabled:', mixed_precision.global_policy())
    except Exception as e:
        print('Mixed precision not enabled:', e)


## 2) Dataset: CIFAR-10


In [ ]:
# =============================
# 2) Load CIFAR-10
# =============================
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype('int32')
y_test = y_test.squeeze().astype('int32')

class_names = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]
num_classes = len(class_names)

print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)


In [ ]:
# Visualize samples
plt.figure(figsize=(10,4))
for i in range(10):
    ax = plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


## 3) Preprocessing + `tf.data`
We resize CIFAR-10 (32×32) to match each backbone input size, then apply the correct `preprocess_input`.


In [ ]:
# =============================
# 3) tf.data pipelines
# =============================
AUTOTUNE = tf.data.AUTOTUNE

from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

augment = keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomContrast(0.1, seed=SEED),
], name='augmentation')


def make_splits(x, y, val_fraction=0.1):
    n = x.shape[0]
    n_val = int(n * val_fraction)
    idx = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(idx)
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]
    return (x[tr_idx], y[tr_idx]), (x[val_idx], y[val_idx])

(x_tr, y_tr), (x_val, y_val) = make_splits(x_train, y_train, val_fraction=0.1)
print('Train split:', x_tr.shape, y_tr.shape)
print('Val split  :', x_val.shape, y_val.shape)


def make_dataset(x, y, image_size, preprocess_fn, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)

    def _map(img, label):
        img = tf.cast(img, tf.float32)
        if training:
            img = augment(img)
        img = tf.image.resize(img, image_size, method='bilinear')
        img = preprocess_fn(img)
        return img, tf.one_hot(label, num_classes)

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    if CACHE_DATASETS:
        ds = ds.cache()
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_resnet_ds = make_dataset(x_tr, y_tr, RESNET_IMAGE_SIZE, resnet_preprocess, training=True)
val_resnet_ds   = make_dataset(x_val, y_val, RESNET_IMAGE_SIZE, resnet_preprocess, training=False)

test_resnet_ds  = make_dataset(x_test, y_test, RESNET_IMAGE_SIZE, resnet_preprocess, training=False)

test_inception_ds  = make_dataset(x_test, y_test, INCEPTION_IMAGE_SIZE, inception_preprocess, training=False)

# sanity
xb, yb = next(iter(train_resnet_ds))
print('Batch:', xb.shape, yb.shape)


## 4) Training utilities


In [ ]:
# =============================
# 4) Utilities
# =============================

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )


def fit_with_timing(model, train_ds, val_ds, epochs, callbacks=None):
    t0 = time.perf_counter()
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks or [],
        verbose=1,
    )
    t1 = time.perf_counter()
    return hist, (t1 - t0)


def freeze_batchnorm(model_or_layer):
    for l in model_or_layer.layers:
        if isinstance(l, tf.keras.layers.BatchNormalization):
            l.trainable = False


def set_finetune(backbone, unfreeze_last_n):
    backbone.trainable = True

    # Freeze all but last N layers
    for l in backbone.layers[:-unfreeze_last_n]:
        l.trainable = False

    # Keep BN frozen
    freeze_batchnorm(backbone)


def save_model_and_size(model, fname):
    path = os.path.join(OUTPUT_DIR, fname)
    model.save(path, include_optimizer=False)
    size_mb = os.path.getsize(path) / (1024**2)
    return path, size_mb


callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
]


## 5) Model A — ResNet50 (transfer learning)


In [ ]:
# =============================
# 5) ResNet50
# =============================
tf.keras.backend.clear_session()
from tensorflow.keras.applications import ResNet50


def build_resnet50(num_classes):
    inputs = keras.Input(shape=(RESNET_IMAGE_SIZE[0], RESNET_IMAGE_SIZE[1], 3))
    backbone = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(RESNET_IMAGE_SIZE[0], RESNET_IMAGE_SIZE[1], 3),
        pooling='avg',
    )
    backbone.trainable = False

    x = backbone(inputs, training=False)
    x = layers.Dropout(0.3)(x)

    # If mixed precision is enabled, force float32 output for stable softmax/loss
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs, name='ResNet50_transfer')
    return model, backbone


resnet_model, resnet_backbone = build_resnet50(num_classes)
resnet_model.summary()


In [ ]:
# Head training
compile_model(resnet_model, lr=1e-3)
resnet_hist_head, resnet_time_head = fit_with_timing(
    resnet_model, train_resnet_ds, val_resnet_ds, EPOCHS_HEAD, callbacks=callbacks
)
print(f'ResNet50 head-training time: {resnet_time_head/60:.2f} minutes')


In [ ]:
# Fine-tuning
if FINE_TUNE:
    set_finetune(resnet_backbone, UNFREEZE_LAST_N_RESNET)
    compile_model(resnet_model, lr=1e-4)
    resnet_hist_ft, resnet_time_ft = fit_with_timing(
        resnet_model, train_resnet_ds, val_resnet_ds, FINE_TUNE_EPOCHS, callbacks=callbacks
    )
    print(f'ResNet50 fine-tuning time: {resnet_time_ft/60:.2f} minutes')
else:
    resnet_hist_ft, resnet_time_ft = None, 0.0


## 6) Model B — InceptionV3 (transfer learning)


In [ ]:
# =============================
# 6) InceptionV3
# =============================
tf.keras.backend.clear_session()
from tensorflow.keras.applications import InceptionV3


def build_inceptionv3(num_classes):
    inputs = keras.Input(shape=(INCEPTION_IMAGE_SIZE[0], INCEPTION_IMAGE_SIZE[1], 3))
    backbone = InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(INCEPTION_IMAGE_SIZE[0], INCEPTION_IMAGE_SIZE[1], 3),
        pooling='avg',
    )
    backbone.trainable = False

    x = backbone(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = keras.Model(inputs, outputs, name='InceptionV3_transfer')
    return model, backbone


train_inception_ds = make_dataset(x_tr, y_tr, INCEPTION_IMAGE_SIZE, inception_preprocess, training=True)
val_inception_ds   = make_dataset(x_val, y_val, INCEPTION_IMAGE_SIZE, inception_preprocess, training=False)

test_inception_ds  = make_dataset(x_test, y_test, INCEPTION_IMAGE_SIZE, inception_preprocess, training=False)

inception_model, inception_backbone = build_inceptionv3(num_classes)
inception_model.summary()


In [ ]:
# Head training
compile_model(inception_model, lr=1e-3)
inception_hist_head, inception_time_head = fit_with_timing(
    inception_model, train_inception_ds, val_inception_ds, EPOCHS_HEAD, callbacks=callbacks
)
print(f'InceptionV3 head-training time: {inception_time_head/60:.2f} minutes')


In [ ]:
# Fine-tuning
if FINE_TUNE:
    set_finetune(inception_backbone, UNFREEZE_LAST_N_INCEPTION)
    compile_model(inception_model, lr=1e-4)
    inception_hist_ft, inception_time_ft = fit_with_timing(
        inception_model, train_inception_ds, val_inception_ds, FINE_TUNE_EPOCHS, callbacks=callbacks
    )
    print(f'InceptionV3 fine-tuning time: {inception_time_ft/60:.2f} minutes')
else:
    inception_hist_ft, inception_time_ft = None, 0.0


## 7) Evaluation + saving (Drive)
This saves models, metrics JSON, and tables/figures to `OUTPUT_DIR`.


In [ ]:
# =============================
# 7) Evaluate + save
# =============================
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(test_resnet_ds, verbose=0)
inc_test_loss, inc_test_acc = inception_model.evaluate(test_inception_ds, verbose=0)

resnet_total_time = float(resnet_time_head + (resnet_time_ft or 0.0))
inception_total_time = float(inception_time_head + (inception_time_ft or 0.0))

resnet_path, resnet_size_mb = save_model_and_size(resnet_model, 'resnet50_cifar10_transfer.keras')
inc_path, inc_size_mb = save_model_and_size(inception_model, 'inceptionv3_cifar10_transfer.keras')

metrics = {
    'resnet50': {
        'test_accuracy': float(resnet_test_acc),
        'test_loss': float(resnet_test_loss),
        'training_time_sec': resnet_total_time,
        'model_size_mb': float(resnet_size_mb),
        'params': int(resnet_model.count_params()),
        'input_size': list(RESNET_IMAGE_SIZE),
        'model_path': resnet_path,
    },
    'inceptionv3': {
        'test_accuracy': float(inc_test_acc),
        'test_loss': float(inc_test_loss),
        'training_time_sec': inception_total_time,
        'model_size_mb': float(inc_size_mb),
        'params': int(inception_model.count_params()),
        'input_size': list(INCEPTION_IMAGE_SIZE),
        'model_path': inc_path,
    },
}

metrics_path = os.path.join(OUTPUT_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('ResNet50  acc/loss:', float(resnet_test_acc), float(resnet_test_loss))
print('Inception acc/loss:', float(inc_test_acc), float(inc_test_loss))
print('Saved:', metrics_path)


## 8) Confusion matrices + classification reports


In [ ]:
# =============================
# 8) Confusion matrices
# =============================

def predict_labels(model, ds):
    y_true, y_pred = [], []
    for bx, by in ds:
        probs = model.predict(bx, verbose=0)
        y_pred.append(np.argmax(probs, axis=1))
        y_true.append(np.argmax(by.numpy(), axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)


def plot_cm(cm, title, filename):
    plt.figure(figsize=(8,6))
    plt.imshow(cm, cmap='Blues')
    plt.title(title)
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha='right')
    plt.yticks(range(num_classes), class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved', path)


r_true, r_pred = predict_labels(resnet_model, test_resnet_ds)
i_true, i_pred = predict_labels(inception_model, test_inception_ds)

resnet_cm = confusion_matrix(r_true, r_pred)
inc_cm = confusion_matrix(i_true, i_pred)

plot_cm(resnet_cm, 'ResNet50 — Confusion Matrix (Test)', 'confusion_matrix_resnet50.png')
plot_cm(inc_cm, 'InceptionV3 — Confusion Matrix (Test)', 'confusion_matrix_inceptionv3.png')

# Save classification reports
resnet_report = classification_report(r_true, r_pred, target_names=class_names, digits=4)
inc_report = classification_report(i_true, i_pred, target_names=class_names, digits=4)

with open(os.path.join(OUTPUT_DIR, 'classification_report_resnet50.txt'), 'w') as f:
    f.write(resnet_report)
with open(os.path.join(OUTPUT_DIR, 'classification_report_inceptionv3.txt'), 'w') as f:
    f.write(inc_report)

print('Saved classification reports to OUTPUT_DIR')


## 9) Curves: accuracy vs epoch, loss vs epoch


In [ ]:
# =============================
# 9) Curves
# =============================

def merge_histories(h1, h2):
    if h2 is None:
        return h1.history
    merged = {k: list(h1.history.get(k, [])) + list(h2.history.get(k, [])) for k in set(h1.history) | set(h2.history)}
    return merged


def plot_curves(hist, title, filename_prefix):
    acc = hist.get('accuracy', [])
    val_acc = hist.get('val_accuracy', [])
    loss = hist.get('loss', [])
    val_loss = hist.get('val_loss', [])
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(epochs, acc, label='Train')
    plt.plot(epochs, val_acc, label='Val')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, loss, label='Train')
    plt.plot(epochs, val_loss, label='Val')
    plt.title(f'{title} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f'{filename_prefix}_accuracy_loss.png')
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved', out)


resnet_hist = merge_histories(resnet_hist_head, resnet_hist_ft)
inc_hist = merge_histories(inception_hist_head, inception_hist_ft)

plot_curves(resnet_hist, 'ResNet50', 'resnet50')
plot_curves(inc_hist, 'InceptionV3', 'inceptionv3')


## 10) Comparison table


In [ ]:
# =============================
# 10) Summary table
# =============================
summary = pd.DataFrame([
    {
        'Model': 'ResNet50',
        'Test Accuracy': float(resnet_test_acc),
        'Test Loss': float(resnet_test_loss),
        'Training Time (min)': resnet_total_time/60,
        'Model Size (MB)': float(resnet_size_mb),
        'Params': int(resnet_model.count_params()),
        'Input Size': f'{RESNET_IMAGE_SIZE[0]}x{RESNET_IMAGE_SIZE[1]}',
    },
    {
        'Model': 'InceptionV3',
        'Test Accuracy': float(inc_test_acc),
        'Test Loss': float(inc_test_loss),
        'Training Time (min)': inception_total_time/60,
        'Model Size (MB)': float(inc_size_mb),
        'Params': int(inception_model.count_params()),
        'Input Size': f'{INCEPTION_IMAGE_SIZE[0]}x{INCEPTION_IMAGE_SIZE[1]}',
    },
])

csv_path = os.path.join(OUTPUT_DIR, 'summary_table.csv')
summary.to_csv(csv_path, index=False)
print('Saved', csv_path)
summary


# Short report (3–5 pages worth of content)
Use this text + your produced graphs/tables as the report.

## 1. Introduction
We compare two major pretrained CNN families via transfer learning:
- **ResNet50**: uses residual connections (skip connections) that help optimize deeper networks.
- **InceptionV3**: uses multi-branch Inception modules to capture multi-scale features efficiently.

## 2. Dataset & preprocessing
We use **CIFAR-10** (10 classes, 32×32 images). Since ImageNet-pretrained models expect larger inputs, we resize:
- ResNet50 → `RESNET_IMAGE_SIZE`
- InceptionV3 → `INCEPTION_IMAGE_SIZE`

We apply lightweight augmentation (flip/contrast) and the correct model-specific preprocessing (`preprocess_input`).

## 3. Transfer learning procedure
1) Freeze backbone and train a new classifier head.
2) Fine-tune: unfreeze only the last N layers with a lower learning rate. BatchNorm layers remain frozen for stability.

## 4. Results
Use `summary_table.csv`, confusion matrices, and classification reports saved to Drive.

## 5. Discussion (why one performed better)
Typical reasons (connect these to your outputs):
- **Architecture**: residual learning vs multi-branch inception features.
- **Input resizing**: CIFAR-10 is low-res; resizing can blur details. Larger resize (e.g., 299×299) increases compute.
- **Compute/time tradeoff**: InceptionV3 often takes longer per epoch.
- **Class confusion**: CIFAR-10 often confuses similar classes (cat vs dog, deer vs horse). Use confusion matrices to justify.

## 6. Conclusion
Conclude based on final test accuracy, training time, and model size (deployment practicality).


## 11) Export standalone report (HTML + Markdown)


In [ ]:
# =============================
# 11) Export report.html + report.md
# =============================
import datetime

def img_tag(fname, caption):
    full = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(full):
        return f'<figure><img src="{fname}" style="max-width:100%;border:1px solid #ddd"/><figcaption>{caption}</figcaption></figure>'
    return f'<p><b>Missing:</b> {fname}</p>'

# Build HTML
stamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
summary_html = summary.to_html(index=False)

html = f'''<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Assignment 03 Report — ResNet50 vs InceptionV3</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; line-height: 1.45; }}
    h1,h2,h3 {{ margin-bottom: 8px; }}
    .meta {{ color: #444; margin-bottom: 18px; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
    th {{ background: #f5f5f5; }}
    figure {{ margin: 16px 0; }}
    figcaption {{ color: #555; font-size: 0.95em; margin-top: 6px; }}
    code {{ background:#f6f6f6; padding:2px 4px; }}
  </style>
</head>
<body>
  <h1>Assignment 03 Report</h1>
  <div class="meta">
    Generated: {stamp}<br>
    Dataset: CIFAR-10<br>
    Models: ResNet50, InceptionV3<br>
    Outputs folder: <code>{OUTPUT_DIR}</code>
  </div>

  <h2>Summary Table</h2>
  {summary_html}

  <h2>Training Curves</h2>
  {img_tag('resnet50_accuracy_loss.png', 'ResNet50 accuracy/loss vs epoch')}
  {img_tag('inceptionv3_accuracy_loss.png', 'InceptionV3 accuracy/loss vs epoch')}

  <h2>Confusion Matrices</h2>
  {img_tag('confusion_matrix_resnet50.png', 'ResNet50 confusion matrix (test)')}
  {img_tag('confusion_matrix_inceptionv3.png', 'InceptionV3 confusion matrix (test)')}

  <h2>Files</h2>
  <ul>
    <li><code>summary_table.csv</code>, <code>metrics.json</code></li>
    <li><code>classification_report_resnet50.txt</code>, <code>classification_report_inceptionv3.txt</code></li>
    <li><code>resnet50_cifar10_transfer.keras</code>, <code>inceptionv3_cifar10_transfer.keras</code></li>
  </ul>

  <h2>Discussion (template)</h2>
  <p>
    Use the summary table and confusion matrices to explain which model performed better.
    Comment on accuracy vs training time vs model size and typical CIFAR-10 confusions.
  </p>
</body>
</html>
'''

html_path = os.path.join(OUTPUT_DIR, 'report.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html)
print('Saved', html_path)

md_path = os.path.join(OUTPUT_DIR, 'report.md')
md_text = f'''# Assignment 03 Report — ResNet50 vs InceptionV3 (CIFAR-10)

Generated: {stamp}

## Outputs folder
`{OUTPUT_DIR}`

## Key artifacts
- `summary_table.csv`
- `metrics.json`
- `resnet50_accuracy_loss.png`, `inceptionv3_accuracy_loss.png`
- `confusion_matrix_resnet50.png`, `confusion_matrix_inceptionv3.png`
- `classification_report_resnet50.txt`, `classification_report_inceptionv3.txt`
- `resnet50_cifar10_transfer.keras`, `inceptionv3_cifar10_transfer.keras`

## Summary
(See `summary_table.csv` or `report.html`.)

## Observations (write based on your outputs)
- Accuracy vs compute tradeoffs
- Why one model converged better
- Which classes are confused and why
'''

with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_text)
print('Saved', md_path)


## 12) Optional: Zip outputs


In [ ]:
import zipfile
zip_path = os.path.join(OUTPUT_DIR, 'assignment03_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUT_DIR):
        for fn in files:
            if fn.endswith('.zip'):
                continue
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, OUTPUT_DIR)
            z.write(full, arcname=rel)
print('Saved', zip_path)
